<a href="https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!git clone https://github.com/DeepakSaini01/ML-WEEK-1.git


fatal: destination path 'ML-WEEK-1' already exists and is not an empty directory.


In [12]:
%cd /content/ML-WEEK-1

/content/ML-WEEK-1


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

ML Task Type: Binary Classification

Reasoning: The goal is to predict whether a specific page URL is entering a performance decline phase (is_declining_label = 1 vs 0). This allows us to automatically flag declining content for editorial review rather than relying on manual performance audits across thousands of URLs.

In [18]:
import pandas as pd
import numpy as np

# Load raw dataset slice from current repository root
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')



In [19]:
# Check exact column names in the dataset
print(list(df.columns))

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [20]:
# Create binary target from trend_direction
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

In [21]:
print("Dataset Shape:", df.shape)
print("\nTarget Class Distribution:")
print(df['is_declining_label'].value_counts(dropna=False))
print("\nTarget Percentage:")
print(df['is_declining_label'].value_counts(normalize=True) * 100)

Dataset Shape: (30000, 45)

Target Class Distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target Percentage:
is_declining_label
1    54.206667
0    45.793333
Name: proportion, dtype: float64


## 2. Target or proxy

Target Variable: is_declining_label (Binary: 1 or 0)

Label Source: Observed outcome. The label is calculated from historical traffic, impression trends, and click changes recorded in search engine log data over time (trend_direction == 'down'), rather than a subjective human rating.

In [23]:
# Check target missing values and summary
print("Missing target values:", df['is_declining_label'].isnull().sum())
print("\nSample Target & Proxy Features Breakdown:")
print(df[['content_id', 'trend_direction', 'is_declining_label']].head(5))

Missing target values: 0

Sample Target & Proxy Features Breakdown:
             content_id trend_direction  is_declining_label
0  content_304f48230142            down                   1
1  content_a1fb4e703a9e            down                   1
2  content_9aa793d4d895            down                   1
3  content_331d6c4de07b          stable                   0
4  content_d99b7a2d90ca            down                   1


## 3. Success metric

Primary Metric: Precision@K (e.g., Precision@50) and ROC-AUC

Defendable Threshold: An ROC-AUC $\ge 0.80$ (or Precision@50 $\ge 85\%$) is solid. In editorial content review, human reviewer time is the key constraint. We need high precision in our top ranked predictions so editors don't waste time auditing pages that aren't actually losing traffic.

In [26]:
from sklearn.metrics import roc_auc_score, precision_score

# Drop any missing target values
y_true = df['is_declining_label'].dropna()

# Use 'trend_pct' (or 'avg_position') as a baseline score for prediction
# Negative trend_pct correlates with declining performance
baseline_scores = -df.loc[y_true.index, 'trend_pct'].fillna(0)

# Compute baseline ROC-AUC
baseline_auc = roc_auc_score(y_true, baseline_scores)
print(f"Baseline Evaluation ROC-AUC Score: {baseline_auc:.4f}")

Baseline Evaluation ROC-AUC Score: 1.0000


## 4. The unit of analysis, as a real dataframe

Unit of Analysis: One row = One unique anonymized URL page record.

In [27]:
# Display unit of analysis structure and key feature columns
print(f"Total Rows (Pages): {len(df):,}")
print(f"Total Features (Columns): {len(df.columns)}")

# Preview core page-level features
features_preview = ['url_id', 'impressions_pre', 'clicks_pre', 'ctr_pre', 'position_pre', 'is_declining_label']
existing_cols = [c for c in features_preview if c in df.columns]
df[existing_cols].head(5)

Total Rows (Pages): 30,000
Total Features (Columns): 45


,is_declining_label
0,1
1,1
2,1
3,0
4,1


## 5. Why ML beats a fixed rule here

A fixed rule (such as if click_drop > 20% and position_change > 3) fails because search traffic is affected by seasonal dynamics, query volume fluctuations, and page age. A minor drop in clicks might be normal seasonality for one page type, but a critical decline for another. Machine learning models capture non-linear relationships across click distributions, CTR drift, and position metrics simultaneously without breaking on edge cases.